# 🔴 HIV Risk Prediction — Tanzania
## Machine Learning Classification Project
**Objective:** Predict whether an individual is at **High Risk** or **Low Risk** of HIV based on behavioral, demographic, and healthcare factors using Logistic Regression and Decision Tree models.

In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)
import pickle

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Libraries imported successfully!")

## 2. Load & Explore the Dataset

In [ ]:
df = pd.read_csv('hiv_risk_dataset.csv')
print("📊 Shape:", df.shape)
print("\n📋 First 5 rows:")
df.head()

In [ ]:
print("📌 Dataset Info:")
df.info()

In [ ]:
print("📈 Statistical Summary (Numeric):")
df.describe()

In [ ]:
print("🔍 Missing Values:")
print(df.isnull().sum())
print("\n✅ No missing values!" if df.isnull().sum().sum() == 0 else "⚠️ Found missing values!")

In [ ]:
print("🎯 Target Variable Distribution:")
print(df['hiv_risk'].value_counts())
print(f"\nHigh Risk: {(df['hiv_risk']=='High Risk').sum()} ({(df['hiv_risk']=='High Risk').mean()*100:.1f}%)")
print(f"Low Risk:  {(df['hiv_risk']=='Low Risk').sum()} ({(df['hiv_risk']=='Low Risk').mean()*100:.1f}%)")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('HIV Risk Prediction — Exploratory Data Analysis', fontsize=15, fontweight='bold')

colors = {'High Risk': '#E53935', 'Low Risk': '#43A047'}

# Target distribution
risk_counts = df['hiv_risk'].value_counts()
axes[0,0].bar(risk_counts.index, risk_counts.values,
              color=[colors[r] for r in risk_counts.index], edgecolor='white', alpha=0.85)
axes[0,0].set_title('HIV Risk Distribution')
axes[0,0].set_ylabel('Count')

# Age distribution by risk
for risk, col in colors.items():
    subset = df[df['hiv_risk'] == risk]['age']
    axes[0,1].hist(subset, bins=20, alpha=0.6, color=col, label=risk)
axes[0,1].set_title('Age Distribution by Risk Level')
axes[0,1].set_xlabel('Age')
axes[0,1].legend()

# Condom use vs Risk
condom_risk = df.groupby(['condom_use','hiv_risk']).size().unstack(fill_value=0)
condom_risk.plot(kind='bar', ax=axes[0,2], color=[colors[c] for c in condom_risk.columns],
                 edgecolor='white', alpha=0.85)
axes[0,2].set_title('Condom Use vs HIV Risk')
axes[0,2].set_xlabel('Condom Use')
axes[0,2].tick_params(axis='x', rotation=15)
axes[0,2].legend(title='Risk')

# Number of partners vs Risk
partner_risk = df.groupby(['number_of_partners','hiv_risk']).size().unstack(fill_value=0)
partner_risk.plot(kind='bar', ax=axes[1,0], color=[colors[c] for c in partner_risk.columns],
                  edgecolor='white', alpha=0.85)
axes[1,0].set_title('Number of Partners vs HIV Risk')
axes[1,0].set_xlabel('Number of Partners')
axes[1,0].tick_params(axis='x', rotation=0)
axes[1,0].legend(title='Risk')

# Partner HIV status vs Risk
partner_status = df.groupby(['partner_hiv_status','hiv_risk']).size().unstack(fill_value=0)
partner_status.plot(kind='bar', ax=axes[1,1], color=[colors[c] for c in partner_status.columns],
                    edgecolor='white', alpha=0.85)
axes[1,1].set_title('Partner HIV Status vs Risk')
axes[1,1].set_xlabel('Partner Status')
axes[1,1].tick_params(axis='x', rotation=15)
axes[1,1].legend(title='Risk')

# HIV Knowledge Score
for risk, col in colors.items():
    subset = df[df['hiv_risk'] == risk]['hiv_knowledge_score']
    axes[1,2].hist(subset, bins=10, alpha=0.6, color=col, label=risk)
axes[1,2].set_title('HIV Knowledge Score by Risk Level')
axes[1,2].set_xlabel('Knowledge Score (0-10)')
axes[1,2].legend()

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA plots saved!")

In [ ]:
# STI history, IV drug use, alcohol use
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Risk Factors vs HIV Risk Level', fontsize=13, fontweight='bold')

for ax, col in zip(axes, ['sti_history', 'iv_drug_use', 'alcohol_use']):
    data = df.groupby([col, 'hiv_risk']).size().unstack(fill_value=0)
    data.plot(kind='bar', ax=ax, color=[colors[c] for c in data.columns],
              edgecolor='white', alpha=0.85)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Risk')

plt.tight_layout()
plt.savefig('risk_factors.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
df_proc = df.copy()

cat_cols = ['gender', 'marital_status', 'education_level', 'employment_status',
            'region', 'residence_type', 'condom_use', 'hiv_tested_before',
            'sti_history', 'blood_transfusion', 'iv_drug_use', 'alcohol_use',
            'healthcare_access', 'partner_hiv_status']

encoders = {}
for col in cat_cols:
    encoders[col] = LabelEncoder()
    df_proc[col] = encoders[col].fit_transform(df_proc[col])
    print(f"✅ Encoded '{col}'")

# Encode target
le_target = LabelEncoder()
df_proc['hiv_risk'] = le_target.fit_transform(df_proc['hiv_risk'])
encoders['hiv_risk'] = le_target
print(f"\n✅ Target classes: {list(le_target.classes_)} → {list(le_target.transform(le_target.classes_))}")
print("   (0 = High Risk, 1 = Low Risk)")

In [ ]:
X = df_proc.drop('hiv_risk', axis=1)
y = df_proc['hiv_risk']

print("Features:", list(X.columns))
print(f"\nDataset: {X.shape[0]} rows × {X.shape[1]} features")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\n✅ Training set: {X_train.shape[0]} samples")
print(f"✅ Testing set:  {X_test.shape[0]} samples")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("\n✅ Features scaled!")

## 5. Model Training & Evaluation

In [ ]:
# ============================================================
# 5A. LOGISTIC REGRESSION
# ============================================================
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)
lr_cv  = cross_val_score(lr, X_train_sc, y_train, cv=5, scoring='accuracy').mean()

print("=" * 48)
print("  📊 LOGISTIC REGRESSION RESULTS")
print("=" * 48)
print(f"  Accuracy : {lr_acc*100:>10.2f}%")
print(f"  ROC-AUC  : {lr_auc:>10.4f}")
print(f"  CV Acc   : {lr_cv*100:>10.2f}%")
print("=" * 48)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=le_target.classes_))

In [ ]:
# ============================================================
# 5B. DECISION TREE CLASSIFIER
# ============================================================
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

dt_acc = accuracy_score(y_test, y_pred_dt)
dt_auc = roc_auc_score(y_test, y_prob_dt)
dt_cv  = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy').mean()

print("=" * 48)
print("  🌳 DECISION TREE RESULTS")
print("=" * 48)
print(f"  Accuracy : {dt_acc*100:>10.2f}%")
print(f"  ROC-AUC  : {dt_auc:>10.4f}")
print(f"  CV Acc   : {dt_cv*100:>10.2f}%")
print("=" * 48)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, target_names=le_target.classes_))

## 6. Model Comparison & Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Model Evaluation: Logistic Regression vs Decision Tree', fontsize=14, fontweight='bold')

# --- Confusion Matrix: LR ---
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
axes[0,0].set_title(f'Logistic Regression\nAccuracy: {lr_acc*100:.2f}%')
axes[0,0].set_ylabel('Actual')
axes[0,0].set_xlabel('Predicted')

# --- Confusion Matrix: DT ---
cm_dt = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[0,1],
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
axes[0,1].set_title(f'Decision Tree\nAccuracy: {dt_acc*100:.2f}%')
axes[0,1].set_ylabel('Actual')
axes[0,1].set_xlabel('Predicted')

# --- ROC Curves ---
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
axes[1,0].plot(fpr_lr, tpr_lr, color='#1976D2', lw=2, label=f'Logistic Reg (AUC={lr_auc:.3f})')
axes[1,0].plot(fpr_dt, tpr_dt, color='#388E3C', lw=2, label=f'Decision Tree (AUC={dt_auc:.3f})')
axes[1,0].plot([0,1],[0,1],'k--', lw=1)
axes[1,0].set_title('ROC Curves Comparison')
axes[1,0].set_xlabel('False Positive Rate')
axes[1,0].set_ylabel('True Positive Rate')
axes[1,0].legend()

# --- Metrics Bar Chart ---
metrics = ['Accuracy', 'ROC-AUC', 'CV Accuracy']
lr_vals = [lr_acc, lr_auc, lr_cv]
dt_vals = [dt_acc, dt_auc, dt_cv]
x = np.arange(len(metrics))
width = 0.35
axes[1,1].bar(x - width/2, lr_vals, width, label='Logistic Regression', color='#1976D2', alpha=0.85)
axes[1,1].bar(x + width/2, dt_vals, width, label='Decision Tree', color='#388E3C', alpha=0.85)
axes[1,1].set_title('Metrics Comparison')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(metrics)
axes[1,1].set_ylim(0, 1.1)
axes[1,1].legend()
for i, (lv, dv) in enumerate(zip(lr_vals, dt_vals)):
    axes[1,1].text(i - width/2, lv + 0.02, f'{lv:.2f}', ha='center', fontsize=9)
    axes[1,1].text(i + width/2, dv + 0.02, f'{dv:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Comparison plots saved!")

In [ ]:
# Feature Importance (Decision Tree)
feat_imp = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
colors_fi = ['#E53935' if v >= feat_imp.quantile(0.75) else '#42A5F5' for v in feat_imp.values]
feat_imp.plot(kind='barh', ax=ax, color=colors_fi)
ax.set_title('Feature Importance — Decision Tree (HIV Risk)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.axvline(feat_imp.mean(), color='orange', linestyle='--', label='Mean importance')
ax.legend()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Feature importance saved!")

## 7. Select Best Model & Save

In [ ]:
print("\n" + "=" * 52)
print("         🏆 MODEL SELECTION SUMMARY")
print("=" * 52)
print(f"{'Metric':<18} {'Logistic Reg':>15} {'Decision Tree':>15}")
print("-" * 52)
print(f"{'Accuracy':<18} {lr_acc*100:>14.2f}% {dt_acc*100:>14.2f}%")
print(f"{'ROC-AUC':<18} {lr_auc:>15.4f} {dt_auc:>15.4f}")
print(f"{'CV Accuracy':<18} {lr_cv*100:>14.2f}% {dt_cv*100:>14.2f}%")
print("=" * 52)

best_model = dt if dt_auc > lr_auc else lr
best_name  = "Decision Tree" if dt_auc > lr_auc else "Logistic Regression"
best_scaler = None if dt_auc > lr_auc else scaler
print(f"\n✅ Best Model: {best_name}")

with open('model.pkl',    'wb') as f: pickle.dump(best_model, f)
with open('encoders.pkl', 'wb') as f: pickle.dump(encoders,   f)
with open('scaler.pkl',   'wb') as f: pickle.dump(scaler,     f)
print("💾 Saved: model.pkl, encoders.pkl, scaler.pkl")

In [ ]:
# Verify
with open('model.pkl', 'rb') as f: loaded = pickle.load(f)
sample_pred = loaded.predict(X_test[:3])
sample_labels = le_target.inverse_transform(sample_pred)
print("✅ Model verified!")
print("\nSample predictions:")
for i, (pred, actual) in enumerate(zip(sample_labels, le_target.inverse_transform(y_test.values[:3]))):
    status = "✅" if pred == actual else "❌"
    print(f"  Sample {i+1}: Predicted = {pred:<12} | Actual = {actual} {status}")

## ✅ Summary

| Item | Details |
|------|---------|
| **Dataset** | 1,000 records, 17 features |
| **Target** | HIV Risk (High Risk / Low Risk) |
| **Models** | Logistic Regression & Decision Tree |
| **Key Risk Factors** | Partner HIV status, condom use, STI history, IV drug use, number of partners |
| **Saved Files** | `model.pkl`, `encoders.pkl`, `scaler.pkl` |